<a href="https://colab.research.google.com/github/wmjx691/rental-market-analyzer/blob/main/scraper_advanced_asset_tracking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### 第一步：環境安裝（Cell 1）

Colab 是 Linux 環境，沒有視窗介面，所以我們必須安裝「無頭模式（Headless）」的瀏覽器驅動。

In [ ]:
# @title 1. 安裝必要套件、瀏覽器驅動與中文字型 (修復方塊字版)
# 安裝 selenium 和 google drive 相關套件
!pip install selenium gspread oauth2client webdriver_manager

# 1. 安裝 Google Chrome
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb
!apt-get install -f -y

# 2. 關鍵修正：安裝中文字型 (解決截圖方塊字問題)
!apt-get install -y fonts-noto-cjk

print("環境安裝完成！中文字型已部署。")

#### 第二步：Google 權限驗證（Cell 2）

這是 Colab 最強大的地方，不用搞複雜的 API Key，直接用你的 Google 帳號登入驗證，就能讓程式控制你的試算表。
*執行時會跳出視窗要求權限，請點選「允許」。*

In [ ]:
# @title 2. Google 帳號授權與試算表連線
from google.colab import auth
import gspread
from google.auth import default

# 進行身分驗證
auth.authenticate_user()       # 這一行會跳出彈窗要你登入
creds, _ = default()           # 這是暫時性的 Session 憑證
gc = gspread.authorize(creds)

print("Google 帳號授權成功！準備開始爬蟲...")

#### 第三步：爬蟲主程式（Cell 3）

這段程式碼會做兩件事：

1. **爬取目標租屋網**（使用無頭模式，因為 Colab 看不到畫面）。
2. **寫入試算表**：如果檔案不存在，它會自動建立一個名為 `TARGET_SHEET_NAME` 的試算表；如果存在，它會把新資料「附加」在最後面。

In [ ]:
# @title 3. 初始化：函式定義與載入歷史資料庫 (Run Once)
import time
import pandas as pd
import re
import pytz
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from IPython.display import Image, display

# --- 全域變數設定 ---
SHEET_NAME = 'TARGET_SHEET_NAME' # 統一設定工作表名稱
TW_TZ = pytz.timezone('Asia/Taipei')

# --- 1. 設定瀏覽器選項 ---
chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--window-size=1920,1080')
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

# --- 2. 工具函式定義 ---

def click_element_by_text(driver, text):
    """點擊含有特定文字的元素"""
    try:
        xpath = f"//label[contains(text(),'{text}')] | //span[contains(text(),'{text}')] | //li[contains(text(),'{text}')]"
        element = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, xpath)))
        driver.execute_script("arguments[0].click();", element)
        time.sleep(0.5)
        return True
    except: return False

def load_history_data():
    """從 Google Sheet 讀取歷史資料並轉換為字典索引"""
    print("📂 正在連線資料庫讀取歷史資料...")
    try:
        sh = gc.open(SHEET_NAME)
        worksheet = sh.sheet1
        data = worksheet.get_all_records()

        if not data:
            print("   -> 資料庫為空，將建立新資料庫。")
            return {}

        df = pd.DataFrame(data)

        # 檢查是否有必要的 ID 欄位
        if '物件ID' not in df.columns:
            print("   -> 舊資料缺少 '物件ID' 欄位，無法進行歷程追蹤。將視為新資料庫。")
            return {}

        # 建立索引：{ '物件ID': {該列所有資料} }
        history_map = {}
        for index, row in df.iterrows():
            # 強制將 ID 轉為字串，避免數字/字串比對錯誤
            pid = str(row['物件ID'])
            history_map[pid] = row.to_dict()

        print(f"✅ 成功載入 {len(history_map)} 筆歷史資產紀錄。")
        return history_map

    except Exception as e:
        print(f"   -> 讀取失敗或是新檔案 (錯誤: {e})，將建立新資料庫。")
        return {}

def save_full_data(df):
    """將 DataFrame 完整覆蓋寫回 Google Sheet"""
    print("💾 正在儲存資料至雲端...")
    try:
        try: sh = gc.open(SHEET_NAME)
        except: sh = gc.create(SHEET_NAME)

        worksheet = sh.sheet1
        worksheet.clear() # 清空舊資料

        # 寫入
        worksheet.append_row(df.columns.tolist())
        worksheet.append_rows(df.values.tolist())

        print(f"✅ 資料庫同步完成！目前管理 {len(df)} 筆資料。")
        print(f"📊 試算表連結: {sh.url}")
    except Exception as e:
        print(f"❌ 儲存失敗: {e}")

# --- 3. 立即執行：預先載入歷史資料 ---
# 這樣做的好處是，我們只需要讀一次，之後在 Cell 4 可以反覆執行爬蟲而不必一直讀取 API
history_database = load_history_data()

In [ ]:
# @title 4. 執行爬蟲：抓取、比對與更新 (Run to Scrape)
# 確保 Cell 3 已經執行過，有 history_database 變數
if 'history_database' not in globals():
    print("⚠️ 請先執行 Cell 3 來載入設定與舊資料！")
else:
    print("🚀 啟動爬蟲代理人...")

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)

    target_url = "https://rental.example.com.tw/?region=17"
    driver.get(target_url)

    try:
        # --- A. 導航與篩選 ---
        try:
            close_btn = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.CSS_SELECTOR, "div.close, i.close, .TIGerm")))
            close_btn.click()
        except: pass

        time.sleep(2)
        print("⚡ 正在設定篩選條件...")
        click_element_by_text(driver, "***HIDDEN_District***")
        click_element_by_text(driver, "***HIDDEN_District***")
        click_element_by_text(driver, "整層住家")

        try:
            search_btn = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(),'搜尋')] | //div[contains(@class,'search')]//button")))
            driver.execute_script("arguments[0].click();", search_btn)
        except: pass

        print("⏳ 等待資料載入中...")
        time.sleep(5)
        # 多捲動幾次以抓取更多資料
        for i in range(5):
            driver.execute_script("window.scrollBy(0, 1000);")
            time.sleep(1.5)

        # --- B. 資料抓取 ---
        items = driver.find_elements(By.CSS_SELECTOR, ".vue-list-rent-item, .listing-recommend-item, div[class*='item']")
        # 過濾太小的雜訊元素
        valid_items = [item for item in items if item.size['height'] > 80]

        print(f"🔍 網頁偵測到 {len(valid_items)} 筆物件卡片，開始解析...")

        current_data_list = []
        today_str = datetime.now(TW_TZ).strftime("%Y-%m-%d")
        last_check_time = datetime.now(TW_TZ).strftime("%Y-%m-%d %H:%M:%S")

        for item in valid_items:
            try:
                full_text = item.text
                if len(full_text) < 10: continue

                # 1. 抓連結與ID
                link = "N/A"
                post_id = "N/A"
                try:
                    link_elm = item.find_element(By.TAG_NAME, "a")
                    link = link_elm.get_attribute("href")
                    id_match = re.search(r'(\d{7,8})', link)
                    if id_match: post_id = id_match.group(1)
                except: pass

                if post_id == "N/A": continue # 無法追蹤 ID 則跳過

                # 2. 抓價格
                price = "N/A"
                price_match = re.search(r'(\d{1,3}(,\d{3})*)\s*元/月', full_text)
                if price_match:
                    price = price_match.group(0).replace("元/月", "").strip()
                else: continue

                # 3. 抓坪數 (過濾 < 20坪)
                area = "N/A"
                area_match = re.search(r'(\d+\.?\d*)\s*坪', full_text)
                if area_match: area = area_match.group(1)
                try:
                    if area != "N/A" and float(area) < 20: continue
                except: pass

                # 4. 抓標題
                lines = full_text.split('\n')
                title = lines[0] if lines else "N/A"
                if len(title) < 5 and len(lines) > 1: title = lines[1]

                # 5. 地址解析
                location_full = "N/A"
                city = "高雄市"
                district = "N/A"
                address_road = "N/A"

                # 簡單邏輯抓取包含"區"的行
                for line in lines:
                    if "區" in line and ("市" in line or "-" in line):
                        location_full = line
                        break

                # 若抓到地址字串，嘗試拆解
                if location_full != "N/A":
                    loc_parts = re.split(r'[-/ ]', location_full)
                    for part in loc_parts:
                        if "區" in part: district = part
                        if "路" in part or "街" in part: address_road = part
                # 補救措施
                if district == "N/A":
                    dist_match = re.search(r'(***HIDDEN_District***|***HIDDEN_District***)', full_text)
                    if dist_match: district = dist_match.group(1)

                # 6. 抓取「網站顯示更新時間」 (New Feature)
                # 目標租屋網上通常會有 "3小時前更新", "昨天", "3天前" 等字樣
                # 這些字樣通常在卡片的底部或某個角落
                site_update_text = "N/A"
                # 嘗試在 full_text 裡找時間關鍵字
                for line in lines:
                    if any(k in line for k in ["更新", "發布", "前", "昨天", "剛"]):
                        # 避免抓到標題裡的字，通常更新時間字數很短
                        if len(line) < 15:
                            site_update_text = line
                            break

                # 7. 歷史比對核心邏輯
                first_seen_date = today_str # 預設今天為首次發現
                days_on_market = 0

                if post_id in history_database:
                    # 舊物件：繼承歷史資料
                    old_record = history_database[post_id]
                    if '首次發現日' in old_record and old_record['首次發現日']:
                        first_seen_date = old_record['首次發現日']
                        try:
                            d1 = datetime.strptime(first_seen_date, "%Y-%m-%d")
                            d2 = datetime.strptime(today_str, "%Y-%m-%d")
                            days_on_market = (d2 - d1).days
                        except: days_on_market = 0

                # 8. 儲存單筆資料
                current_data_list.append({
                    "物件ID": post_id,
                    "最後更新時間": last_check_time,
                    "首次發現日": first_seen_date,
                    "上架已持續天數": days_on_market,
                    "網站顯示更新": site_update_text, # 新增欄位
                    "標題": title,
                    "價格": price,
                    "坪數": area,
                    "縣市/行政區": f"{city}/{district}",
                    "路段/地址": address_road,
                    "完整顯示地址": location_full,
                    "連結": link
                })

            except Exception as e: continue

        # --- C. 資料合併與存檔 ---
        if not current_data_list:
            print("⚠️ 未抓取到符合條件的資料。")
        else:
            df_new = pd.DataFrame(current_data_list)

            # 計算本次抓取中的廣告重複度
            df_new['廣告投放數'] = df_new.groupby(['價格', '坪數'])['物件ID'].transform('count')

            # 準備合併：先複製舊資料庫
            final_data_map = history_database.copy()

            # 用新資料更新舊資料 (相同的 ID 會被新的狀態覆蓋，但首次發現日已在上方邏輯保留)
            for index, row in df_new.iterrows():
                pid = row['物件ID']
                final_data_map[pid] = row.to_dict()

            # 轉回 DataFrame
            df_final = pd.DataFrame(list(final_data_map.values()))

            # 排序：按照最後更新時間 (新抓到的放上面)
            if '最後更新時間' in df_final.columns:
                df_final = df_final.sort_values(by='最後更新時間', ascending=False)

            # 指定欄位順序 (美觀用)
            target_cols = ["物件ID", "最後更新時間", "首次發現日", "上架已持續天數", "網站顯示更新", "標題", "價格", "坪數", "廣告投放數", "縣市/行政區", "路段/地址", "完整顯示地址", "連結"]
            # 只選取存在的欄位
            final_cols = [c for c in target_cols if c in df_final.columns]
            df_final = df_final[final_cols]

            # 執行存檔
            save_full_data(df_final)

            # 更新記憶體中的歷史資料庫，以防使用者想立刻再跑一次
            history_database = final_data_map

    except Exception as e:
        print(f"❌ 發生錯誤: {str(e)}")
        driver.save_screenshot('error_run.png')
        display(Image('error_run.png'))

    driver.quit()

---

### 如何執行你的「一週測試計畫」

既然你要測試一週，且每三天執行一次，操作流程如下：

1. **第一次（今天）：**
* 登入你的 Google Drive，建立一個 Colab 筆記本。
* 將上述三段代碼貼入。
* 依序點擊「播放鍵」執行 Cell 1, 2, 3, 4。
* 執行完後，去你的 Google Drive 根目錄找找看，會有一個 **`TARGET_SHEET_NAME`** 的試算表。打開來確認資料是否正確。


2. **第二次（三天後）：**
* 打開這個 Colab 網頁。
* **重要：** 因為 Colab 會重置環境，所以你必須**再次點擊 Cell 1, 2, 3, 4**。
* 程式會自動把新的資料「新增」到那張試算表的下面，不會覆蓋舊資料。


3. **第三次（六天後）：**
* 重複上述動作。



### 提醒（關於目標租屋網的反爬蟲）

在 Colab 的「無頭模式（Headless）」下，瀏覽器特徵非常明顯，租屋網這種網站有時會直接阻擋（你可能會看到程式跑完但說「抓到 0 筆物件」）。

* **如果發生這種情況**：代表目標租屋網擋掉了 Colab 的 IP 或特徵。這時候最簡單的解法，還是回到我一開始提供的 **PC 本地端執行**（因為你在本地有視窗介面，比較像真人）。